In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

os.environ["TORCHDYNAMO_INLINE_INBUILT_NN_MODULES"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MUJOCO_GL"] = "egl"


import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from tensordict import TensorDict, from_module

torch.autograd.set_detect_anomaly(True)
torch.set_float32_matmul_precision("high")

from fast_td3.fast_td3_utils import (
    EmpiricalNormalization,
    StructuredEmpiricalNormalization
)

from fast_td3.actors import ActorEGNN, ActorEGNN_V2, Actor

In [ ]:
from fast_td3.hyperparams import HumanoidBenchArgs

robot = "h1"

args = HumanoidBenchArgs(
    env_name=f"{robot}-balance_simple-v1",
    total_timesteps=50000,
    render_interval=5000,
    eval_interval=1000,
    num_envs=16,
    batch_size=8192,
    actor_hidden_dim=384,
)

In [ ]:
amp_enabled = args.amp and args.cuda and torch.cuda.is_available()
amp_device_type = (
    "cuda"
    if args.cuda and torch.cuda.is_available()
    else "mps" if args.cuda and torch.backends.mps.is_available() else "cpu"
)
amp_dtype = torch.bfloat16 if args.amp_dtype == "bf16" else torch.float16

scaler = GradScaler(enabled=amp_enabled and amp_dtype == torch.float16)


random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.backends.cudnn.deterministic = args.torch_deterministic

if not args.cuda:
    device = torch.device("cpu")
else:
    if torch.cuda.is_available():
        device = torch.device(f"cuda:{args.device_rank}")
    elif torch.backends.mps.is_available():
        device = torch.device(f"mps:{args.device_rank}")
    else:
        raise ValueError("No GPU available")
print(f"Using device: {device}")

In [ ]:
from fast_td3.environments.humanoid_bench_env import HumanoidBenchEnv

env_type = "humanoid_bench"
eval_envs = HumanoidBenchEnv(args.env_name, 1, device=device)
render_env = HumanoidBenchEnv(args.env_name, 1, render_mode="rgb_array", device=device)

n_act = eval_envs.num_actions
n_obs = eval_envs.num_obs if type(eval_envs.num_obs) == int else eval_envs.num_obs[0]
if eval_envs.asymmetric_obs:
    n_critic_obs = (
        eval_envs.num_privileged_obs
        if type(eval_envs.num_privileged_obs) == int
        else eval_envs.num_privileged_obs[0]
    )
else:
    n_critic_obs = n_obs
action_low, action_high = -1.0, 1.0

In [ ]:
checkpoint_path = None
checkpoint_path = "./models/egnn_v2_h1-balance_simple-v1_16envs_1000001steps_7726fe_70000.pt"
obs_normalizer = StructuredEmpiricalNormalization(env_name=args.env_name, device=device)
# Actor setup
actor = ActorEGNN_V2(
    num_envs=16,
    batch_size=args.batch_size,
    device=device,
    hidden_dim=64,
    n_layers=4,
    act_fn="relu",
    robot=robot,
    env_name=args.env_name,
	  tanh=True,
	  coords_agg="sum",
)


torch_checkpoint = torch.load(
    f"{checkpoint_path}", map_location=device, weights_only=False
)
obs_normalizer.load_state_dict(torch_checkpoint["obs_normalizer_state"])
pretrained_state_dict = torch_checkpoint["actor_state_dict"]    
actor.load_state_dict(torch_checkpoint["actor_state_dict"])

normalize_obs = obs_normalizer.forward

In [ ]:
# checkpoint_path = "./models/mlp_h1-balance_simple-v0_16envs_1000001steps_aef324_final.pt"
# obs_normalizer = EmpiricalNormalization(shape=n_obs, device=device)
# xanchor_normalizer = nn.Identity()
# normalize_obs = obs_normalizer.forward
# normalize_xanchor = xanchor_normalizer.forward

# actor = Actor(
#     n_obs=n_obs,
#     n_act=n_act,
#     num_envs=args.num_envs,
#     device=device,
#     init_scale=args.init_scale,
#     hidden_dim=args.actor_hidden_dim,
# )

# torch_checkpoint = torch.load(
#     f"{checkpoint_path}", map_location=device, weights_only=False
# )
# obs_normalizer.load_state_dict(torch_checkpoint["obs_normalizer_state"])
# # xanchor_normalizer.load_state_dict(torch_checkpoint["xanchor_normalizer_state"])
# pretrained_state_dict = torch_checkpoint["actor_state_dict"]    
# actor.load_state_dict(torch_checkpoint["actor_state_dict"])

# normalize_obs = obs_normalizer.forward
# normalize_xanchor = xanchor_normalizer.forward

# print("actor parameters:", sum(p.numel() for p in actor.parameters()))

In [ ]:
def evaluate():
    obs_normalizer.eval()
    num_eval_envs = eval_envs.num_envs
    episode_returns = torch.zeros(num_eval_envs, device=device)
    episode_lengths = torch.zeros(num_eval_envs, device=device)
    done_masks = torch.zeros(num_eval_envs, dtype=torch.bool, device=device)
    
    obs = eval_envs.reset(random_position=False, random_orientation=False)
    # for _ in range(eval_envs.max_episode_steps):
    for _ in range(10):
        with torch.no_grad(), autocast(
            device_type=amp_device_type, dtype=amp_dtype, enabled=amp_enabled
        ):  
            obs = normalize_obs(obs)
            actions = actor(obs)

        next_obs, rewards, dones, _  = eval_envs.step(actions.float())
        episode_returns = torch.where(
            ~done_masks, episode_returns + rewards, episode_returns
        )
        episode_lengths = torch.where(~done_masks, episode_lengths + 1, episode_lengths)
        done_masks = torch.logical_or(done_masks, dones)
        if done_masks.all():
            break
        obs = next_obs

    obs_normalizer.train()
    return episode_returns.mean().item(), episode_lengths.mean().item()

In [ ]:
evaluate()

In [ ]:
import tempfile
import imageio
import base64
from IPython.display import display, HTML


def frames_to_video_html(frames, fps=30):
	"""
	Convert a list of numpy arrays to an HTML5 video element.

	Args:
		frames (list): List of numpy arrays representing video frames
		fps (int): Frames per second for the video

	Returns:
		HTML object containing the video element
	"""
	# Create a temporary file to store the video
	with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as temp_file:
		temp_filename = temp_file.name

	# Save frames as video
	imageio.mimsave(temp_filename, frames, fps=fps)

	# Read the video file and encode it to base64
	with open(temp_filename, "rb") as f:
		video_data = f.read()
	video_b64 = base64.b64encode(video_data).decode("utf-8")

	# Create HTML video element
	video_html = f"""
	<video width="640" height="480" controls>
		<source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
		Your browser does not support the video tag.
	</video>
	"""

	# Clean up the temporary file
	os.unlink(temp_filename)

	return HTML(video_html)


def render_with_rollout():
	obs_normalizer.eval()

	# Quick rollout for rendering
	obs = render_env.reset(random_position=False, random_orientation=True)
	renders = []

	qpos_max = []
	qvel_max = []
	qpos_min = []
	qvel_min = []
	object_vel_max = []
	object_vel_min = []


	for i in range(render_env.max_episode_steps):
		with torch.no_grad(), autocast(
			device_type=amp_device_type, dtype=amp_dtype, enabled=amp_enabled
		):
			qpos_max.append(obs[:, 7:26].max().item())
			qvel_max.append(obs[:, 32:].max().item())
			qpos_min.append(obs[:, 7:26].min().item())
			qvel_min.append(obs[:, 32:].min().item())
			object_vel_max.append(obs[:, 26:32].max().item())
			object_vel_min.append(obs[:, 26:32].min().item())
   
			obs = normalize_obs(obs)   
			actions = actor(obs)

		next_obs, _, done, _ = render_env.step(actions.float())
		if i % 2 == 0:
			if env_type == "humanoid_bench":
				renders.append(render_env.render())
			else:
				renders.append(render_env.state)
		if done.any():
			break
		obs = next_obs

	if env_type == "mujoco_playground":
		renders = render_env.render_trajectory(renders)

	obs_normalizer.train()
	video_html = frames_to_video_html(renders, fps=30)
	display(video_html)

	# import matplotlib.pyplot as plt
	# plt.plot(qpos_max)
	# plt.plot(qvel_max)
	# plt.plot(qpos_min)
	# plt.plot(qvel_min)
	# plt.show()

	# print(max(qpos_max))
	# print(min(qpos_min))
	# print(max(qvel_max))
	# print(min(qvel_min))
	# print(max(object_vel_max))
	# print(min(object_vel_min))
 
render_with_rollout()